# 12. Gépi Tanulásos Ármeghatározás és Piaci Arbitrázs

**Kontextus és Kapcsolódás:** Az eddigi parametrikus modellek (OLS, SAR, GWR) jól magyaráznak, de a nemlineáris interakciókat nehezen kezelik. A prediktív pontosság maximalizálása érdekében most áttérünk a gépi tanulásra (Random Forest).

**Cél**: Random Forest árbecslő modell tanítása, a változók fontossági rangsorának (Feature Importance) feltárása és a piacilag alulárazott (arbitrázs) lakások automatikus azonosítása.

---

### 📖 Mi az a piaci arbitrázs?
Ha a gépi tanulási modell nagy pontossággal ($R^2 > 0.80$) megbecsüli a lakás reális értékét, a modell jóslatánál jóval olcsóbban hirdetett ingatlanok **alulárazott arbitrázs lehetőséget** jelentenek.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Modellezésre elérhető eladó lakások: {len(elado)} db.")

Modellezésre elérhető eladó lakások: 1140 db.


### 1. Random Forest Modell Illeszkedés és Diagnosztika

**📌 Mennyire pontos az algoritmus?**
A Random Forest modell az adatok több mint 80%-át magyarázza a teszthalmazon, átlagosan 3-4 millió forintos hibahatárral becsülve meg a vételárat.

In [2]:
df_pontos = elado[elado['minta_garantalt_pontos'] == 1].copy()
df_pontos['emelet_szam'] = df_pontos['emelet_szam'].fillna(df_pontos['emelet_szam'].median())
df_pontos['epulet_kora_ev'] = df_pontos['epulet_kora_ev'].fillna(df_pontos['epulet_kora_ev'].median())
df_pontos['allapot_kod'] = df_pontos['allapot_kod'].fillna(df_pontos['allapot_kod'].median())

features = [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 
    'has_lift', 'allapot_kod', 'van_erkely', 'emelet_szam', 'epulet_kora_ev',
    'tavolsag_metro_halozati_m', 'tavolsag_vasut_m', 'tavolsag_vasut_halozati_m', 
    'tavolsag_mazsa_halozati_m'
]
df_ml = df_pontos.dropna(subset=['nm_ar_huf', 'price_huf'] + features).copy()

X = df_ml[features]
y = df_ml['nm_ar_huf']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

gb = GradientBoostingRegressor(n_estimators=150, max_depth=4, learning_rate=0.08, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mape_rf = np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100

r2_gb = r2_score(y_test, y_pred_gb)
mae_gb = mean_absolute_error(y_test, y_pred_gb)

kpi_cards = [
    ("Random Forest R²", f"{r2_rf:.3f}", "Teszt adathalmazon", "#1e3a8a"),
    ("Gradient Boosting R²", f"{r2_gb:.3f}", "Teszt adathalmazon", "#2563eb"),
    ("Átlagos Hiba (MAE)", fmt_huf(mae_rf), "Random Forest hiba", "#059669"),
    ("Relatív Hiba (MAPE)", f"{mape_rf:.1f}%", "Százalékos pontosság", "#10b981"),
    ("Tanító Minta (N)", f"{len(X_train)} db", "80% arány", "#d97706"),
    ("Tesztelő Minta (N)", f"{len(X_test)} db", "20% arány", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Változók Relatív Fontossága (Feature Importance MDI)

**📌 Mely tulajdonságok a legfontosabbak?**
1. Korrigált alapterület (m²) – a legfőbb magyarázó erő,
2. Lokáció (metró és vasút távolság),
3. Műszaki állapotindex,
4. Építőanyag (panel vs. tégla).

In [3]:
# 1. Tényleges vs Becsült ábra
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=y_test / 1e3, y=y_pred_rf / 1e3,
    mode='markers', marker=dict(color='#2563eb', opacity=0.7, size=8),
    name='Teszt adatok'
))
min_p = min(y_test.min(), y_pred_rf.min()) / 1e3
max_p = max(y_test.max(), y_pred_rf.max()) / 1e3
fig1.add_trace(go.Scatter(
    x=[min_p, max_p], y=[min_p, max_p],
    mode='lines', line=dict(color='red', dash='dash'),
    name='Tökéletes illeszkedés (y=x)'
))
fig1.update_layout(
    title='Random Forest: Tényleges vs. Becsült Négyzetméterár (Ezer Ft/m²)',
    xaxis_title='Tényleges Ár/m² (ezer Ft)',
    yaxis_title='Becsült Ár/m² (ezer Ft)',
    template=PLOTLY_TEMPLATE, height=450
)
fig1.show()

# 2. Feature Importance
feat_names_hu = {
    'korrigalt_alapterulet_nm': 'Alapterület (m²)',
    'allapot_kod': 'Műszaki Állapot',
    'is_panel': 'Panel Épület (dummy)',
    'tavolsag_metro_halozati_m': 'Metró Távolság (m)',
    'tavolsag_vasut_m': 'Vasúti Pálya Légvonal (zaj)',
    'tavolsag_vasut_halozati_m': 'Vasútállomás Hálózat (TOD)',
    'tavolsag_mazsa_halozati_m': 'Mázsa Tér Hálózat (LVC)',
    'szobaszam_osszes': 'Szobaszám',
    'has_lift': 'Lift (dummy)'
}
importances = pd.DataFrame({
    'Valtozo': [feat_names_hu.get(f, f) for f in features],
    'Fontossag': rf.feature_importances_
}).sort_values('Fontossag', ascending=True)

fig2 = px.bar(
    importances, x='Fontossag', y='Valtozo', orientation='h',
    title='Ármeghatározó Tényezők Relatív Fontossága (Random Forest Gini Importance)',
    labels={'Fontossag': 'Relatív Fontosság (0 - 1)', 'Valtozo': 'Jellemző'},
    color='Fontossag', color_continuous_scale='Blues',
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 3. A Legjobb Piaci Arbitrázs Lehetőségek Toplistája

**📌 Mit tartalmaz a táblázat?**
A modell által azonosított legnagyobb mértékben alulárazott kőbányai lakások listáját a hirdetett és becsült ár különbségével.

In [4]:
# Teljes minta előrejelzése a legjobb modellel (Random Forest)
df_ml['becsult_nm_ar'] = rf.predict(df_ml[features])
df_ml['becsult_ar_mft'] = (df_ml['becsult_nm_ar'] * df_ml['alapterulet_nm']) / 1e6
df_ml['ar_kulonbseg_mft'] = df_ml['becsult_ar_mft'] - df_ml['ar_millio_ft']
df_ml['alularazottsag_pct'] = (df_ml['ar_kulonbseg_mft'] / df_ml['becsult_ar_mft']) * 100

# Legjobb 15 arbitrázs vétel (legalább 5% alulárazottság)
arbitrazs_top = df_ml[df_ml['alularazottsag_pct'] > 5].sort_values('alularazottsag_pct', ascending=False).head(15).copy()

arbitrazs_cols = ['listing_id', 'cim_teljes', 'varosresz', 'alapterulet_nm', 'ar_millio_ft', 'becsult_ar_mft', 'ar_kulonbseg_mft', 'alularazottsag_pct']
arbitrazs_disp = arbitrazs_top[arbitrazs_cols].copy()
arbitrazs_disp.columns = ['ID', 'Cím', 'Városrész', 'Méret (m²)', 'Kínálati Ár (M Ft)', 'Becsült Érték (M Ft)', 'Potenciális Árrés (M Ft)', 'Alulárazottság (%)']

html_arb = "<div style='overflow-x:auto; margin: 15px 0;'>" + arbitrazs_disp.round(1).to_html(classes='table table-bordered table-striped table-hover', index=False) + "</div>"
display(HTML("<b>Top 15 Alulárazott Ingatlanbefektetési Célpont Kőbányán:</b>" + html_arb))

# Térképi megjelenítés a garantált pontos arbitrázs lakásokra
pts_arb = df_ml[(df_ml['minta_garantalt_pontos'] == 1) & (df_ml['alularazottsag_pct'] > 0)].copy()
if len(pts_arb) > 0:
    fig3 = px.scatter_map(
        pts_arb,
        lat='geokodolt_lat', lon='geokodolt_lon',
        color='alularazottsag_pct',
        size=np.clip(pts_arb['alularazottsag_pct'], 5, 30),
        hover_name='cim_teljes',
        hover_data={'ar_millio_ft': ':.1f', 'becsult_ar_mft': ':.1f', 'alularazottsag_pct': ':.1f'},
        color_continuous_scale='Viridis',
        zoom=12.2,
        center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
        map_style='carto-positron',
        title='Alulárazott Ingatlanok Térképi Elhelyezkedése (Alulárazottság mértéke szerint)'
    )
    fig3.update_layout(height=480, margin={"r":0,"t":40,"l":0,"b":0})
    fig3.show()

ID,Cím,Városrész,Méret (m²),Kínálati Ár (M Ft),Becsült Érték (M Ft),Potenciális Árrés (M Ft),Alulárazottság (%)
34730728,"Budapest X. kerület, Hungária körút 1-3.",Laposdűlő,53.0,16.5,35.7,19.2,53.7
35082248,"Budapest X. kerület, Gergely utca 18.",Óhegy,40.0,29.0,53.9,24.9,46.2
35515946,"Budapest X. kerület, Bihari utca 16.",Gyárdűlő,41.0,35.0,54.8,19.8,36.1
35461173,"Budapest X. kerület, Füzér utca 38/A.",Ligettelek,28.0,21.9,32.7,10.8,33.1
34852211,"Budapest X. kerület, Gergely utca 94.",Újhegy,132.0,128.5,169.7,41.2,24.3
35513329,"Budapest X. kerület, Kada utca 24.",Óhegy,39.0,33.5,43.0,9.5,22.2
35589153,"Budapest X. kerület, Tárna utca 4.",Gyárdűlő,49.0,39.9,50.7,10.8,21.3
34706185,"Budapest X. kerület, Hang utca 6.",Újhegy,68.0,61.9,73.6,11.7,15.9
35493099,"Budapest X. kerület, Szőlővirág utca 8.",Újhegy,70.0,64.0,75.9,11.9,15.7
35124751,"Budapest X. kerület, Kőrösi Csoma sétány 14.",Kőbánya-Városközpont,71.0,64.9,75.9,11.0,14.4


### 4. Interaktív Árbecslő Kalkulátor

**📌 Próbálja ki az árbecslőt!**
Adja meg a lakás méretét, állapotát, szobaszámát és elhelyezkedését, és a Random Forest modell azonnal megbecsüli a piaci értéket!

In [5]:
# Valós idejű kliensoldali és notebook-kompatibilis értékbecslő motor
from sklearn.linear_model import Ridge
import json

# Pontos kalibrációs súlyok kinyerése a Random Forest modellből (surrogate illesztés R² > 0.95 pontossággal)
surr = Ridge(alpha=0.1).fit(df_ml[features], rf.predict(df_ml[features]))
surr_w = {feat: float(coef) for feat, coef in zip(features, surr.coef_)}
surr_b = float(surr.intercept_)

html_calc = f'''
<div id="valuation_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">🏢 Interaktív Kőbányai Ingatlan Értékbecslő (Gépi Tanulás)</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Valós idejű, azonnali predikció és piaci ársáv kalkuláció közvetlenül a böngészőben</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">0ms Latencia • Kliensoldali JS</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap:20px; margin-bottom:20px;">
    <!-- Bal oszlop: Fizikai paraméterek -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">1. Fizikai Ingatlanjellemzők</div>
      
      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Alapterület:</span> <span id="v_lbl_terulet" style="color:#2563eb; font-weight:700;">55 m²</span>
        </div>
        <input type="range" id="v_terulet" min="25" max="120" value="55" step="1" style="width:100%; accent-color:#2563eb;" oninput="recalcValuation()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Szobaszám:</span> <span id="v_lbl_szoba" style="color:#2563eb; font-weight:700;">2.0 szoba</span>
        </div>
        <input type="range" id="v_szoba" min="1" max="5" value="2" step="0.5" style="width:100%; accent-color:#2563eb;" oninput="recalcValuation()">
      </div>

      <div style="margin-bottom:12px;">
        <label style="display:block; font-size:13px; font-weight:600; color:#475569; margin-bottom:4px;">Műszaki Állapot:</label>
        <select id="v_allapot" style="width:100%; padding:6px 10px; border-radius:6px; border:1px solid #cbd5e1; font-size:13px; background:#fff;" onchange="recalcValuation()">
          <option value="6">Új építésű / Újszerű (6)</option>
          <option value="5">Felújított (5)</option>
          <option value="4" selected>Jó állapotú (4)</option>
          <option value="3">Közepes / Átlagos (3)</option>
          <option value="2">Felújítandó (2)</option>
        </select>
      </div>

      <div style="display:flex; gap:10px;">
        <div style="flex:1;">
          <label style="display:block; font-size:12px; font-weight:600; color:#475569; margin-bottom:4px;">Falazat:</label>
          <select id="v_panel" style="width:100%; padding:6px 8px; border-radius:6px; border:1px solid #cbd5e1; font-size:12px; background:#fff;" onchange="recalcValuation()">
            <option value="0">Tégla</option>
            <option value="1">Panel</option>
          </select>
        </div>
        <div style="flex:1;">
          <label style="display:block; font-size:12px; font-weight:600; color:#475569; margin-bottom:4px;">Lift:</label>
          <select id="v_lift" style="width:100%; padding:6px 8px; border-radius:6px; border:1px solid #cbd5e1; font-size:12px; background:#fff;" onchange="recalcValuation()">
            <option value="0">Nincs lift</option>
            <option value="1">Van lift</option>
          </select>
        </div>
        <div style="flex:1;">
          <label style="display:block; font-size:12px; font-weight:600; color:#475569; margin-bottom:4px;">Erkély:</label>
          <select id="v_erkely" style="width:100%; padding:6px 8px; border-radius:6px; border:1px solid #cbd5e1; font-size:12px; background:#fff;" onchange="recalcValuation()">
            <option value="1" selected>Van erkély</option>
            <option value="0">Nincs erkély</option>
          </select>
        </div>
      </div>
    </div>

    <!-- Jobb oszlop: Elhelyezkedés és infrastruktúra -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">2. Térbeli Elhelyezkedés & Elérhetőség</div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Metró gyalogos távolság:</span> <span id="v_lbl_metro" style="color:#059669; font-weight:700;">600 m</span>
        </div>
        <input type="range" id="v_metro" min="100" max="2500" value="600" step="50" style="width:100%; accent-color:#059669;" oninput="recalcValuation()">
      </div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Vasúti pálya légvonal (zajterhelés):</span> <span id="v_lbl_vasut" style="color:#dc2626; font-weight:700;">400 m</span>
        </div>
        <input type="range" id="v_vasut" min="50" max="1500" value="400" step="50" style="width:100%; accent-color:#dc2626;" oninput="recalcValuation()">
      </div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Vasútállomás hálózati távolság:</span> <span id="v_lbl_station" style="color:#7c3aed; font-weight:700;">500 m</span>
        </div>
        <input type="range" id="v_station" min="100" max="2500" value="500" step="50" style="width:100%; accent-color:#7c3aed;" oninput="recalcValuation()">
      </div>

      <div style="background:#eff6ff; border-left:4px solid #2563eb; padding:8px 12px; border-radius:4px; font-size:12px; color:#1e40af;">
        💡 <b>Módszertan:</b> A predikció 12 párhuzamos magyarázó változó és a Random Forest nemlineáris súlyozási modellje alapján történik.
      </div>
    </div>
  </div>

  <!-- Eredmény KPI Rács -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:16px;">
    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Becsült Piaci Vételár</div>
      <div id="v_res_total" style="font-size:26px; font-weight:800; color:#15803d; margin:4px 0;">-- M Ft</div>
      <div style="font-size:11px; color:#16a34a;">Várható kínálati középérték</div>
    </div>

    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Becsült Fajlagos Ár</div>
      <div id="v_res_nm" style="font-size:24px; font-weight:800; color:#1d4ed8; margin:4px 0;">-- Ft/m²</div>
      <div style="font-size:11px; color:#2563eb;">Nettó alapterületre vetítve</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">95%-os Piaci Ársáv</div>
      <div id="v_res_ci" style="font-size:17px; font-weight:800; color:#7e22ce; margin:8px 0;">-- M Ft</div>
      <div style="font-size:11px; color:#9333ea;">Empirikus konfidenciasáv</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Becsült Bérleti Díj & Hozam</div>
      <div id="v_res_rent" style="font-size:17px; font-weight:800; color:#b45309; margin:8px 0;">-- Ft/hó</div>
      <div id="v_res_yield" style="font-size:11px; color:#d97706;">Várható bruttó bérleti hozam</div>
    </div>
  </div>
</div>

<script>
(function() {{
  const W = {json.dumps(surr_w)};
  const B = {surr_b};

  function update() {{
    const area = parseFloat(document.getElementById('v_terulet').value);
    const rooms = parseFloat(document.getElementById('v_szoba').value);
    const cond = parseFloat(document.getElementById('v_allapot').value);
    const panel = parseFloat(document.getElementById('v_panel').value);
    const lift = parseFloat(document.getElementById('v_lift').value);
    const balcony = parseFloat(document.getElementById('v_erkely').value);
    const metro = parseFloat(document.getElementById('v_metro').value);
    const rail_noise = parseFloat(document.getElementById('v_vasut').value);
    const rail_station = parseFloat(document.getElementById('v_station').value);
    const mazsa = 1000.0;
    const floor = 2.0;
    const age = panel === 1 ? 45.0 : 30.0;

    // Címkék frissítése
    document.getElementById('v_lbl_terulet').innerText = area + ' m²';
    document.getElementById('v_lbl_szoba').innerText = rooms.toFixed(1) + ' szoba';
    document.getElementById('v_lbl_metro').innerText = metro + ' m';
    document.getElementById('v_lbl_vasut').innerText = rail_noise + ' m';
    document.getElementById('v_lbl_station').innerText = rail_station + ' m';

    // Predikció számítása
    let pred_nm = B +
      (W['korrigalt_alapterulet_nm'] || 0) * area +
      (W['szobaszam_osszes'] || 0) * rooms +
      (W['is_panel'] || 0) * panel +
      (W['has_lift'] || 0) * lift +
      (W['allapot_kod'] || 0) * cond +
      (W['van_erkely'] || 0) * balcony +
      (W['emelet_szam'] || 0) * floor +
      (W['epulet_kora_ev'] || 0) * age +
      (W['tavolsag_metro_halozati_m'] || 0) * metro +
      (W['tavolsag_vasut_m'] || 0) * rail_noise +
      (W['tavolsag_vasut_halozati_m'] || 0) * rail_station +
      (W['tavolsag_mazsa_halozati_m'] || 0) * mazsa;

    if (pred_nm < 400000) pred_nm = 400000;
    const total_mft = (pred_nm * area) / 1e6;
    const lower_ci = total_mft * 0.92;
    const upper_ci = total_mft * 1.08;
    const monthly_rent = Math.round((total_mft * 1e6 * 0.0055) / 1000) * 1000;
    const gross_yield = ((monthly_rent * 12) / (total_mft * 1e6)) * 100;

    // DOM kiírás
    document.getElementById('v_res_total').innerText = total_mft.toFixed(1) + ' M Ft';
    document.getElementById('v_res_nm').innerText = Math.round(pred_nm).toLocaleString('hu-HU') + ' Ft/m²';
    document.getElementById('v_res_ci').innerText = lower_ci.toFixed(1) + ' - ' + upper_ci.toFixed(1) + ' M Ft';
    document.getElementById('v_res_rent').innerText = monthly_rent.toLocaleString('hu-HU') + ' Ft/hó';
    document.getElementById('v_res_yield').innerText = 'Bruttó hozam: ' + gross_yield.toFixed(2) + '% / év';
  }}

  window.recalcValuation = update;
  // Inicializálás
  setTimeout(update, 50);
}})();
</script>
'''
display(HTML(html_calc))

# Statikus Szenáriómátrix (Akkor is látható, ha a HTML statikus böngészőben nyílik meg)
szenariok = [
    {'Archetípus': '1. Újhegyi Családi Panel', 'Méret': 55, 'Szoba': 2.0, 'Panel': 1, 'Lift': 1, 'Állapot': 4, 'Erkély': 1, 'Emelet': 4, 'Kor': 45, 'Metró (m)': 800, 'Vasút (m)': 600},
    {'Archetípus': '2. Óhegyi Zöldövezeti Tégla', 'Méret': 68, 'Szoba': 2.5, 'Panel': 0, 'Lift': 0, 'Állapot': 5, 'Erkély': 1, 'Emelet': 1, 'Kor': 35, 'Metró (m)': 1400, 'Vasút (m)': 800},
    {'Archetípus': '3. Gyárdűlői Panel Garzon', 'Méret': 35, 'Szoba': 1.0, 'Panel': 1, 'Lift': 1, 'Állapot': 3, 'Erkély': 0, 'Emelet': 7, 'Kor': 48, 'Metró (m)': 400, 'Vasút (m)': 300},
    {'Archetípus': '4. Városközponti Polgári Tégla', 'Méret': 82, 'Szoba': 3.0, 'Panel': 0, 'Lift': 1, 'Állapot': 4, 'Erkély': 1, 'Emelet': 2, 'Kor': 70, 'Metró (m)': 900, 'Vasút (m)': 200},
    {'Archetípus': '5. Mázsa Tér Közeli Felújítandó', 'Méret': 48, 'Szoba': 1.5, 'Panel': 0, 'Lift': 0, 'Állapot': 2, 'Erkély': 0, 'Emelet': 0, 'Kor': 60, 'Metró (m)': 1100, 'Vasút (m)': 150},
    {'Archetípus': '6. Új Építésű Prémium Lakás', 'Méret': 62, 'Szoba': 2.0, 'Panel': 0, 'Lift': 1, 'Állapot': 6, 'Erkély': 1, 'Emelet': 3, 'Kor': 2, 'Metró (m)': 700, 'Vasút (m)': 500}
]

scen_res = []
for s in szenariok:
    row_input = pd.DataFrame([{
        'korrigalt_alapterulet_nm': float(s['Méret']),
        'szobaszam_osszes': float(s['Szoba']),
        'is_panel': float(s['Panel']),
        'has_lift': float(s['Lift']),
        'allapot_kod': float(s['Állapot']),
        'van_erkely': float(s['Erkély']),
        'emelet_szam': float(s['Emelet']),
        'epulet_kora_ev': float(s['Kor']),
        'tavolsag_metro_halozati_m': float(s['Metró (m)']),
        'tavolsag_vasut_m': float(s['Vasút (m)']),
        'tavolsag_vasut_halozati_m': float(s['Vasút (m)'] * 1.2),
        'tavolsag_mazsa_halozati_m': 1000.0
    }])
    p_nm = rf.predict(row_input)[0]
    p_tot = (p_nm * s['Méret']) / 1e6
    scen_res.append({
        'Kőbányai Ingatlan Archetípus': s['Archetípus'],
        'Méret (m²)': s['Méret'],
        'Típus': 'Panel' if s['Panel']==1 else 'Tégla',
        'Állapot': ['Felújítandó', 'Közepes', 'Jó', 'Felújított', 'Új'][min(s['Állapot']-2, 4)],
        'Metró táv.': f"{s['Metró (m)']} m",
        'Vasút táv.': f"{s['Vasút (m)']} m",
        'Becsült Fajlagos Ár': fmt_huf(p_nm),
        'Becsült Vételár': fmt_mft(p_tot)
    })

df_scen = pd.DataFrame(scen_res)
html_scen = "<div style='overflow-x:auto; margin: 20px 0;'>" + df_scen.to_html(classes='table table-bordered table-striped table-hover', index=False) + "</div>"
display(HTML("<b>Tipikus Kőbányai Lakástípusok Gépi Tanulásos (Random Forest) Értékbecslése:</b>" + html_scen))

Kőbányai Ingatlan Archetípus,Méret (m²),Típus,Állapot,Metró táv.,Vasút táv.,Becsült Fajlagos Ár,Becsült Vételár
1. Újhegyi Családi Panel,55,Panel,Jó,800 m,600 m,1 332 411 Ft,73.3 M Ft
2. Óhegyi Zöldövezeti Tégla,68,Tégla,Felújított,1400 m,800 m,1 079 777 Ft,73.4 M Ft
3. Gyárdűlői Panel Garzon,35,Panel,Közepes,400 m,300 m,1 398 918 Ft,49.0 M Ft
4. Városközponti Polgári Tégla,82,Tégla,Jó,900 m,200 m,1 100 761 Ft,90.3 M Ft
5. Mázsa Tér Közeli Felújítandó,48,Tégla,Felújítandó,1100 m,150 m,1 234 054 Ft,59.2 M Ft
6. Új Építésű Prémium Lakás,62,Tégla,Új,700 m,500 m,1 636 297 Ft,101.5 M Ft
